<a href="https://colab.research.google.com/github/hangkimdiep-boop/RL_E1405_Submited_2026/blob/main/MICROGRID_ENERGY_OPTIMIZATION_PPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**MICROGRID ENERGY OPTIMIZATION - PPO (Proximal Policy Optimization)**

Author: Hang Diep Kim

Date: February 2026

**1️⃣ Install Dependencies & Imports**

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
import matplotlib.pyplot as plt
import json, os, time
from torch.distributions import Categorical
from collections import deque
from typing import Dict, List, Tuple, Optional


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")


🖥️ Using device: cuda


**2️⃣ PPO Configuration**



*   PPO HYPERPARAMETERS



In [2]:
CONFIG = {
    # SETUP ENVIRONMENT
    "battery_capacity": 100,
    "battery_efficiency": 0.95,
    "max_charge_rate": 20,
    "max_discharge_rate": 20,
    "max_solar": 50,
    "max_wind": 30,
    "base_demand": 40,
    "demand_std": 10,
    "grid_price_max": 0.25,
    "grid_price_min": 0.05,
    "hours_per_episode": 24,

    # PPO SPECIFIC
    "state_dim": 8, # State Space (8D): battery, solar, wind, price, hour_sin, hour_cos, prev_action, demand
    "action_dim": 5,# Action Space (5): discharge, charge_renewable, buy_grid, renew + discharge, renew + grid

    # NETWORK ARCHITECTURE
    "hidden_dims": [128, 128],

    # LEARNING RATE
    "lr_actor": 3e-4,
    "lr_critic": 1e-3,

    # PPO PARAMETERS
    "gamma": 0.99,               # Discount factor
    "gae_lambda": 0.95,          # GAE lambda
    "clip_epsilon": 0.2,         # PPO clipping range
    "ppo_epochs": 12,            # Update epochs per rollout
    "mini_batch_size": 64,       # Mini-batch size
    "entropy_coeff": 0.01,       # Entropy bonus
    "value_loss_coeff": 0.5,     # Value loss weight
    "max_grad_norm": 0.5,        # Gradient clipping

    # TRAINING
    "num_episodes": 1500,
    "max_steps_per_episode": 24, # 1 episode = 1 day (24h, step 1h)
    "rollout_steps": 96,         # Steps before update (4 episodes)
    "log_freq": 10,

    # RANDOM SEED
    "seed": 42,

    # REWARD
    "reward_renewable": 1.0,
    "reward_grid_penalty": -2.0,
    "reward_unmet_penalty": -5.0,
    "reward_battery_wear": -0.1,
    "reward_peak_bonus": 0.5,

    # TERMINATION
    "battery_critical_low": 0.05,
    "battery_critical_high": 1.0,
    "max_unmet_ratio": 0.20,
}

print("✅ PPO Config loaded!")
print(f"   LR Actor: {CONFIG['lr_actor']}, LR Critic: {CONFIG['lr_critic']}")
print(f"   Clip ε: {CONFIG['clip_epsilon']}, PPO Epochs: {CONFIG['ppo_epochs']}")
print(f"   GAE λ: {CONFIG['gae_lambda']}, Entropy: {CONFIG['entropy_coeff']}")

✅ PPO Config loaded!
   LR Actor: 0.0003, LR Critic: 0.001
   Clip ε: 0.2, PPO Epochs: 12
   GAE λ: 0.95, Entropy: 0.01


**3️⃣ Microgrid Environment**

In [ ]:


class MicrogridEnv:
    def __init__(self, config: Dict):
        self.config = config
        self.battery_capacity = config["battery_capacity"]
        self.battery_efficiency = config["battery_efficiency"]
        self.max_charge = config["max_charge_rate"]
        self.max_discharge = config["max_discharge_rate"]
        self.max_solar = config["max_solar"]
        self.max_wind = config["max_wind"]
        self.base_demand = config["base_demand"]
        self.demand_std = config["demand_std"]
        self.grid_price_min = config["grid_price_min"]
        self.grid_price_max = config["grid_price_max"]
        self.hours_per_episode = config["hours_per_episode"]
        self.r_renewable = config["reward_renewable"]
        self.r_grid = config["reward_grid_penalty"]
        self.r_unmet = config["reward_unmet_penalty"]
        self.r_wear = config["reward_battery_wear"]
        self.r_bonus = config["reward_peak_bonus"]
        self.battery_critical_low = config.get("battery_critical_low", 0.05)
        self.battery_critical_high = config.get("battery_critical_high", 1.0)
        self.max_unmet_ratio = config.get("max_unmet_ratio", 0.20)
        self.reset()

    def reset(self, seed=None):
        if seed is not None:
            np.random.seed(seed)
        self.battery_level = self.battery_capacity * 0.5
        self.current_hour = 0
        self.prev_action = 0
        self.total_demand = 0.0
        self.total_grid_cost = 0.0
        self.total_renewable_used = 0.0
        self.total_unmet = 0.0
        self.episode_history = []
        return self._get_obs()

    def _get_obs(self):
        d = self._get_demand(self.current_hour)
        p = self._get_price(self.current_hour)
        s = self._get_solar(self.current_hour)
        w = self._get_wind(self.current_hour)

        obs = np.array([
            self.battery_level / self.battery_capacity,
            d / (self.base_demand * 2),
            s / self.max_solar,
            w / self.max_wind,
            (p - self.grid_price_min) / (self.grid_price_max - self.grid_price_min),
            (np.sin(2 * np.pi * self.current_hour / 24) + 1) / 2,
            (np.cos(2 * np.pi * self.current_hour / 24) + 1) / 2,
            self.prev_action / 4.0,
        ], dtype=np.float32)
        return np.clip(obs, 0.0, 1.0)

    def _get_demand(self, hour):
        mp = np.exp(-((hour - 8) ** 2) / 8)
        ep = np.exp(-((hour - 19) ** 2) / 8)
        base = self.base_demand * (0.5 + 0.3 * mp + 0.4 * ep)
        return max(0, base + np.random.normal(0, self.demand_std * 0.3))

    def _get_solar(self, hour):
        if 6 <= hour <= 18:
            base = self.max_solar * np.sin(np.pi * (hour - 6) / 12)
            return max(0, base * np.random.uniform(0.8, 1.2))
        return 0.0

    def _get_wind(self, hour):
        base = self.max_wind * 0.5
        var = self.max_wind * 0.5 * np.sin(np.pi * hour / 12)
        return max(0, (base + var) * np.random.uniform(0.5, 1.5))

    def _get_price(self, hour):
        if 7 <= hour <= 9 or 18 <= hour <= 21:
            base = self.grid_price_max
        elif 22 <= hour or hour <= 6:
            base = self.grid_price_min
        else:
            base = (self.grid_price_min + self.grid_price_max) / 2
        return base * np.random.uniform(0.9, 1.1)

    def step(self, action):
        demand = self._get_demand(self.current_hour)
        solar = self._get_solar(self.current_hour)
        wind = self._get_wind(self.current_hour)
        price = self._get_price(self.current_hour)
        renewable = solar + wind

        renewable_used = grid_purchased = battery_charge = battery_discharge = unmet_demand = 0.0

        if action == 0:  # Discharge
            discharge = min(self.battery_level, self.max_discharge, demand)
            battery_discharge = discharge
            self.battery_level -= discharge
            remaining = demand - discharge * self.battery_efficiency
            if remaining > 0: unmet_demand = remaining
        elif action == 1:  # Charge from renewable
            supply = min(renewable, demand)
            renewable_used = supply
            remaining = demand - supply
            if remaining > 0: unmet_demand = remaining
            excess = renewable - supply
            if excess > 0:
                charge = min(excess, self.max_charge, self.battery_capacity - self.battery_level)
                battery_charge = charge
                self.battery_level += charge * self.battery_efficiency
        elif action == 2:  # Buy from grid
            grid_purchased = demand
        elif action == 3:  # Renewable + Discharge
            renewable_used = min(renewable, demand)
            remaining = demand - renewable_used
            if remaining > 0:
                discharge = min(self.battery_level, self.max_discharge, remaining)
                battery_discharge = discharge
                self.battery_level -= discharge
                remaining -= discharge * self.battery_efficiency
            if remaining > 0: unmet_demand = remaining
        elif action == 4:  # Renewable + Grid
            renewable_used = min(renewable, demand)
            remaining = demand - renewable_used
            if remaining > 0: grid_purchased = remaining

        # Reward
        norm_price = (price - self.grid_price_min) / (self.grid_price_max - self.grid_price_min)
        is_peak = 18 <= self.current_hour <= 21
        reward = (
            self.r_renewable * (renewable_used / self.base_demand)
            + self.r_grid * (grid_purchased / self.base_demand) * norm_price
            + self.r_unmet * (unmet_demand / self.base_demand)
            + self.r_wear * ((battery_charge + battery_discharge) / self.max_charge)
        )
        if is_peak and grid_purchased == 0:
            reward += self.r_bonus

        self.total_demand += demand
        self.total_renewable_used += renewable_used
        self.total_grid_cost += grid_purchased * price
        self.total_unmet += unmet_demand
        self.episode_history.append({
            "hour": self.current_hour, "demand": demand, "solar": solar,
            "wind": wind, "price": price, "action": action,
            "renewable_used": renewable_used, "grid_purchased": grid_purchased,
            "battery_level": self.battery_level, "reward": reward,
        })

        self.current_hour += 1
        self.prev_action = action

        done = self.current_hour >= self.hours_per_episode
        bat_ratio = self.battery_level / self.battery_capacity
        if bat_ratio < self.battery_critical_low: done = True
        if self.total_demand > 0 and self.total_unmet / self.total_demand > self.max_unmet_ratio:
            done = True

        info = {
            "total_cost": self.total_grid_cost,
            "renewable_ratio": self.total_renewable_used / max(1, self.total_demand),
            "unmet_ratio": self.total_unmet / max(1, self.total_demand),
        }
        return self._get_obs(), reward, done, info

print("✅ MicrogridEnv defined!")